In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import tiktoken
from datasets import load_dataset

import time
from components.dataset import TextDataset
from components.model import GPTModel
from components.tokenizer import tokenizer as final_tokenizer

/home/software/Documents/projects/thonk-v1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
base_encoding= tiktoken.get_encoding("cl100k_base")
special_tokens={
  "[INST]": base_encoding.n_vocab,
  "[/INST]": base_encoding.n_vocab + 1,
  "[STARTOFTEXT]": base_encoding.n_vocab + 2,
  "[ENDOFTEXT]": base_encoding.n_vocab + 3,
}
tokenizer = tiktoken.Encoding(
  name="bob",
  pat_str=base_encoding._pat_str,
  mergeable_ranks=base_encoding._mergeable_ranks,
  special_tokens={**base_encoding._special_tokens, **special_tokens}
)
def encode(text):
  return tokenizer.encode(text, allowed_special={"[INST]", "[/INST]", "[STARTOFTEXT]", "[ENDOFTEXT]"})

def decode(text):
  return tokenizer.decode(text)
text="[STARTOFTEXT] hello world [ENDOFTEXT]"
print(encode(text))
print(decode(encode(text)))

[100279, 24748, 1917, 220, 100280]
[STARTOFTEXT] hello world [ENDOFTEXT]


In [3]:
base_dataset = load_dataset("HuggingFaceFW/fineweb-edu", "default", streaming=True)
next_text = next(iter(base_dataset['train']['text']))

In [4]:
block_size = 256
n_embedding = 256
n_layers = 8
n_heads = 8
dropout_p = 0.1

In [5]:
dataset = TextDataset(base_dataset, block_size)

for i in range(10):
  start = time.perf_counter()
  x = (next(iter(dataset['train'])))
  end = time.perf_counter()
  # print(f"time {i}: {end-start}")

In [6]:
model = GPTModel(block_size=block_size, n_embedding=n_embedding, n_layers=n_layers, n_heads=n_heads, dropout_p=dropout_p, vocab_size=final_tokenizer.n_vocab)
text_ctx = torch.tensor(encode("hello!")).unsqueeze(0)
model(text_ctx)

tensor([[[-0.1710,  0.2729, -0.1331,  ...,  0.4879,  0.7135, -0.7988],
         [ 0.5039,  0.4128,  2.2392,  ...,  0.1177, -0.5716,  0.5351]]],
       grad_fn=<UnsafeViewBackward0>)